In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import torch
import torch.nn.functional as torch_functional
import json
import os
import math

# Verificar que los archivos existan
FICHERO_PESOS = "modelo_pesos.pth"
FICHERO_VOCAB = "vocabulario.json"

if not os.path.exists(FICHERO_PESOS) or not os.path.exists(FICHERO_VOCAB):
    print("⚠️ Error: No se encuentran 'modelo_pesos.pth' o 'vocabulario.json' en el directorio actual.")
else:
    # Cargar vocabulario si no está cargado
    with open(FICHERO_VOCAB, "r", encoding="utf-8") as f:
        caracteres = json.load(f)["caracteres"]
    
    vocab_size = len(caracteres)
    char_a_int = {ch: i for i, ch in enumerate(caracteres)}
    int_a_char = {i: ch for i, ch in enumerate(caracteres)}
    
    encode = lambda s: [char_a_int[c] for c in s if c in char_a_int]
    decode = lambda l: "".join([int_a_char[i] for i in l])

    # Componentes visuales interactivos con ipywidgets
    prompt_input = widgets.Text(
        value='Don Quijote',
        description='Prompt:',
        disabled=False,
        layout=widgets.Layout(width='400px')
    )
    
    slider_chars = widgets.IntSlider(
        value=300,
        min=50,
        max=1000,
        step=50,
        description='Caracteres:',
    )
    
    slider_temp = widgets.FloatSlider(
        value=1.0,
        min=0.1,
        max=2.0,
        step=0.1,
        description='Temperatura:',
    )
    
    boton_generar = widgets.Button(
        description='Generar Texto',
        button_style='primary',
        icon='robot'
    )
    
     salida_texto = widgets.Output()

    def on_button_clicked(b):
        with salida_texto:
            clear_output()
            prompt = prompt_input.value
            tokens_iniciales = encode(prompt)
            
            if not tokens_iniciales:
                print("⚠️ Los caracteres ingresados no existen en el vocabulario del modelo.")
                return
            
            print("Generando texto...")
            contexto = torch.tensor([tokens_iniciales], dtype=torch.long, device=device)
            
            with torch.no_grad():
                for _ in range(slider_chars.value):
                    cond = contexto[:, -max_seq_len:]
                    logits = modelo(cond)
                    # Aplicar temperatura
                    logits = logits[:, -1, :] / slider_temp.value
                    probs = torch_functional.softmax(logits, dim=-1)
                    siguiente_char = torch.multinomial(probs, num_samples=1)
                    contexto = torch.cat((contexto, siguiente_char), dim=1)
            
            resultado = decode(contexto[0].tolist())
            clear_output()
            print("--- Texto Generado ---")
            print(resultado)

    boton_generar.on_click(on_button_clicked)

    # Mostrar la interfaz en el notebook
    display(widgets.VBox([prompt_input, slider_chars, slider_temp, boton_generar, salida_texto]))